# PC vs lncRNA — Extended Element & Structure Statistics

Descriptive statistics of genomic **and RNA-structural** features for protein-coding (PC)
and lncRNA transcripts in the **common-CDHIT** set. Extends `000_element_statistics.ipynb`
with the revision features, all read from a single combined table:

- **Repeats — RNA (spliced)**: RepeatMasker on the mature transcript (`rna_*`) — reproduces the
  original *Rep (Total)* row.
- **Repeats — DNA (genomic)**: RepeatMasker on the unspliced locus (`dna_*`) — *new*.
- **Non-B DNA (NBD, total)**: nBMST motif totals.
- **rG4 peaks**: rG4detector RNA G-quadruplex peaks (`rg4_*`) — *new* (count + % with peak; no coverage metric).
- **ScanFold**: predicted RNA secondary-structure thermodynamics (avgMFE / avgZ / avgED / paired / competition) — *new* (structural metrics, not counts/coverage).

**Data source (single file)**
`revision-wt/results/gencode.v47.common.cdhit.cv/features/supplementary_features.tsv`

NOTE: Statistics are calculated from all values (including transcripts were feature X = 0). There is a sanity check verifying that there are non empty values coerced to NaN and silently dropped.


## 1. Imports and Setup


In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

DATASET = "gencode.v47.common.cdhit.cv"
BASE = Path("/mnt/cbib/LNClassifier/paper")
# revision-wt worktree holds the combined feature table (see worktree-routing).
SUPP_FEATURES = BASE / "revision-wt/results" / DATASET / "features/supplementary_features.tsv"
# transcript_length lives in the REP feature table (not in the combined file).
REP_FEATURES  = BASE / "te_pipeline/results/te_analysis_flexible/features/all_transcripts_te_features.corrected.csv"

TABLE_OUT = BASE / "revision-wt/results" / DATASET / "supplementary_tables" / "element_statistics_tables"
TABLE_OUT.mkdir(parents=True, exist_ok=True)

CLASS_ORDER  = ["coding", "lncRNA"]
CLASS_LABELS = {"coding": "Protein-coding", "lncRNA": "lncRNA"}

print("Supp features :", SUPP_FEATURES)
print("Table output  :", TABLE_OUT)


Supp features : /mnt/cbib/LNClassifier/paper/revision-wt/results/gencode.v47.common.cdhit.cv/features/supplementary_features.tsv
Table output  : /mnt/cbib/LNClassifier/paper/revision-wt/results/gencode.v47.common.cdhit.cv/supplementary_tables/element_statistics_tables


## 2. Helper functions


In [ ]:
def _ms(s, nd=2):
    v = pd.to_numeric(s, errors="coerce").dropna()
    return f"{v.mean():.{nd}f} ± {v.std():.{nd}f}" if len(v) else "NA"


def _mad_str(s, nd=0):
    v = pd.to_numeric(s, errors="coerce").dropna()
    m = v.median()
    return f"{m:.{nd}f} ± {(v - m).abs().median():.{nd}f}" if len(v) else "NA"


def _vda(u, n1, n2):
    return round(u / (n1 * n2), 4)


def _stats_row(group, feat, pc_cnt, lnc_cnt, pc_cov=None, lnc_cov=None):
    """One feature row: counts + optional coverage mean±std for PC/lncRNA, with MWU+VDA."""
    pc_c = pd.to_numeric(pc_cnt, errors="coerce").dropna()
    lnc_c = pd.to_numeric(lnc_cnt, errors="coerce").dropna()
    n1, n2 = len(pc_c), len(lnc_c)
    row = {
        "Group": group,
        "Feature": feat,
        "PC counts (mean±std)": _ms(pc_c),
        "PC % coverage (mean±std)": "NA",
        "lncRNA counts (mean±std)": _ms(lnc_c),
        "lncRNA % coverage (mean±std)": "NA",
        "PC % with hit": round((pc_c > 0).mean() * 100, 1) if n1 else np.nan,
        "lncRNA % with hit": round((lnc_c > 0).mean() * 100, 1) if n2 else np.nan,
        "VDA (count)": np.nan,
        "VDA (cov)": np.nan,
        "p_MWU_count": np.nan,
        "p_MWU_cov": np.nan,
    }
    if n1 and n2:
        u, p = mannwhitneyu(pc_c, lnc_c, alternative="two-sided")
        row.update({"VDA (count)": _vda(u, n1, n2), "p_MWU_count": p})
    if pc_cov is not None and lnc_cov is not None:
        pc_v = pd.to_numeric(pc_cov, errors="coerce").dropna()
        lnc_v = pd.to_numeric(lnc_cov, errors="coerce").dropna()
        if len(pc_v) and len(lnc_v):
            row["PC % coverage (mean±std)"] = _ms(pc_v)
            row["lncRNA % coverage (mean±std)"] = _ms(lnc_v)
            u_v, p_v = mannwhitneyu(pc_v, lnc_v, alternative="two-sided")
            row.update({"VDA (cov)": _vda(u_v, len(pc_v), len(lnc_v)), "p_MWU_cov": p_v})
    return row


def _apply_bh_fdr(df, p_cols):
    """BH-FDR pooled over p_cols; adds *_adj columns."""
    vals, locs = [], []
    for col in p_cols:
        for i, v in df[col].items():
            if pd.notna(v):
                vals.append(v)
                locs.append((i, col))
    if vals:
        _, pa, _, _ = multipletests(vals, method="fdr_bh")
        for col in p_cols:
            df[f"{col}_adj"] = np.nan
        for (i, col), a in zip(locs, pa):
            df.at[i, f"{col}_adj"] = a
    else:
        for col in p_cols:
            df[f"{col}_adj"] = np.nan
    return df


def desc_row(label, cnt_col, cov_col=None, extra=None):
    """One feature row for the summary Table 2: counts + coverage mean±std, with MWU+VDA."""
    pc_c  = pd.to_numeric(pc[cnt_col],  errors="coerce").dropna()
    lnc_c = pd.to_numeric(lnc[cnt_col], errors="coerce").dropna()
    u_c, p_c = mannwhitneyu(pc_c, lnc_c, alternative="two-sided")
    row = {
        "Feature": label,
        "PC counts (mean±std)":         _ms(pc_c),
        "PC % coverage (mean±std)":     "—",
        "lncRNA counts (mean±std)":     _ms(lnc_c),
        "lncRNA % coverage (mean±std)": "—",
        "PC % with hit":     round((pc_c > 0).mean() * 100, 1),
        "lncRNA % with hit": round((lnc_c > 0).mean() * 100, 1),
        "VDA (count)": _vda(u_c, len(pc_c), len(lnc_c)),
        "p_MWU_count": p_c,
        "p_MWU_cov":   np.nan,
    }
    if cov_col is not None:
        pc_v  = pd.to_numeric(pc[cov_col],  errors="coerce").dropna()
        lnc_v = pd.to_numeric(lnc[cov_col], errors="coerce").dropna()
        u_v, p_v = mannwhitneyu(pc_v, lnc_v, alternative="two-sided")
        row["PC % coverage (mean±std)"]     = _ms(pc_v)
        row["lncRNA % coverage (mean±std)"] = _ms(lnc_v)
        row["VDA (cov)"] = _vda(u_v, len(pc_v), len(lnc_v))
        row["p_MWU_cov"] = p_v
    if extra:
        row.update(extra)
    return row


## 3. Load combined feature table

The file is large (~0.5 GB); read only the columns we need.


In [ ]:
# Read the live combined feature table by feature family rather than a stale hand-written
# subset: the revision TSV has many additional repeat and motif columns required
# by the extended aggregate table.
supp_cols = pd.read_csv(SUPP_FEATURES, sep="\t", nrows=0).columns
feature_prefixes = (
    "rna_", "dna_", "apr_", "dr_", "gq_plus_", "gq_minus_", "ir_", "mr_", "str_", "tri_", "z_",
    "all_nonb_", "any_nonb_", "motif_", "total_nonb_", "rg4_", "avg", "frac", "n_",
)
USE_COLS = [c for c in supp_cols if c == "seq_ID" or c.startswith(feature_prefixes) or c in {"has_peak", "motif_types_present", "any_nonb_present"}]

# The combined revision TSV no longer contains the legacy metadata columns; derive
# coding_class from the project binary class table instead.
binary_table = BASE / "revision-wt/results" / DATASET / "tables" / f"{DATASET}_binary_class_table.tsv"
labels = (
    pd.read_csv(binary_table, sep="\t", usecols=["seq_ID", "real"])
      .set_index("seq_ID")["real"]
      .map({True: "coding", False: "lncRNA"})
      .rename("coding_class")
)

df = pd.read_csv(SUPP_FEATURES, sep="\t", usecols=USE_COLS).set_index("seq_ID")
df["coding_class"] = labels.reindex(df.index)
df["coding_class"] = df["coding_class"].str.strip()

# Attach spliced transcript_length from the REP table.
length = (pd.read_csv(REP_FEATURES, usecols=["transcript_id", "transcript_length"])
            .set_index("transcript_id")["transcript_length"])
df["transcript_length"] = length.reindex(df.index)

pc  = df[df["coding_class"] == "coding"]
lnc = df[df["coding_class"] == "lncRNA"]
print(f"{len(df):,} transcripts  |  PC {len(pc):,}  lncRNA {len(lnc):,}")
df[["coding_class", "transcript_length", "rna_global_rm_count",
    "dna_global_rm_count", "rg4_peak_count", "avgMFE_mean"]].head(3)



111,652 transcripts  |  PC 66,376  lncRNA 45,276


,coding_class,transcript_length,rna_global_rm_count,dna_global_rm_count,rg4_peak_count,avgMFE_mean
seq_ID,,,,,,
ENST00000000412.8,coding,2450,1,7,0,-29.158951
ENST00000002596.6,coding,7160,6,56,0,-27.612589
ENST00000002829.8,coding,3607,2,34,1,-41.944106


In [ ]:
# Diagnostic: count values that were not missing before numeric coercion but became NaN.
feature_cols = [c for c in df.columns if c not in {"coding_class", "transcript_length"}]
coerced_to_nan = (
    df[feature_cols]
      .apply(lambda s: (pd.to_numeric(s, errors="coerce").isna() & s.notna()).sum())
      .sort_values(ascending=False)
)
coerced_to_nan = coerced_to_nan[coerced_to_nan > 0]
print(f"Numeric coercions to NaN: {int(coerced_to_nan.sum())} total values across {len(coerced_to_nan)} columns")
display(coerced_to_nan.head(20))


Numeric coercions to NaN: 0 total values across 0 columns


Series([], dtype: int64)

## 4. Descriptive statistics table (Table 2)


Format follows the original Table 2: **Counts (mean ± std)** and **% coverage (mean ± std)**
per class, plus Mann–Whitney U test with Vargha–Delaney A (VDA) effect size and pooled BH-FDR.
Coverage is `—` where a feature has no coverage metric (rG4 peaks).


In [ ]:
rows = [
    desc_row("Repeats — RNA (spliced)",  "rna_global_rm_count", "rna_global_rm_total_length_pct"),
    desc_row("Repeats — DNA (genomic)",  "dna_global_rm_count", "dna_global_rm_total_length_pct"),
    desc_row("Non-B DNA (total)",        "total_nonb_count",    "total_nonb_coverage_pct"),
    desc_row("rG4 peaks",                "rg4_peak_count",      None,
             extra={"PC % with hit": round(pc["has_peak"].astype(bool).mean()*100, 1),
                    "lncRNA % with hit": round(lnc["has_peak"].astype(bool).mean()*100, 1)}),
]
desc = pd.DataFrame(rows)

# Pooled BH-FDR over every MWU p-value (counts + coverage), mapped back by position.
flat = list(desc["p_MWU_count"]) + list(desc["p_MWU_cov"])
mask = [pd.notna(v) for v in flat]
adj = np.full(len(flat), np.nan)
adj[mask] = multipletests([v for v in flat if pd.notna(v)], method="fdr_bh")[1]
n = len(desc)
desc["p_MWU_count_adj"] = adj[:n]
desc["p_MWU_cov_adj"]   = adj[n:]

# Transcript-length reference row (median ± MAD, no counts/coverage).
len_row = pd.DataFrame([{
    "Feature": "Transcript length (median±MAD)",
    "PC counts (mean±std)": _mad_str(pc["transcript_length"]),
    "lncRNA counts (mean±std)": _mad_str(lnc["transcript_length"]),
}])

desc_out = pd.concat([len_row, desc], ignore_index=True)
display(desc_out.set_index("Feature").round(4))


,PC counts (mean±std),lncRNA counts (mean±std),PC % coverage (mean±std),lncRNA % coverage (mean±std),PC % with hit,lncRNA % with hit,VDA (count),p_MWU_count,p_MWU_cov,VDA (cov),p_MWU_count_adj,p_MWU_cov_adj
Feature,,,,,,,,,,,,
Transcript length (median±MAD),1620 ± 1012,1028 ± 457,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Repeats — RNA (spliced),1.50 ± 2.55,2.27 ± 2.39,7.12 ± 11.57,28.27 ± 24.35,56.9,81.4,0.3499,0.0,0.0,0.2274,0.0,0.0
Repeats — DNA (genomic),101.97 ± 187.69,75.68 ± 140.19,36.57 ± 17.98,41.27 ± 21.06,96.9,95.6,0.5889,0.0,0.0,0.4320,0.0,0.0
Non-B DNA (total),288.02 ± 551.34,215.73 ± 415.33,13.63 ± 16.79,13.10 ± 26.96,99.9,99.3,0.6066,0.0,0.0,0.5446,0.0,0.0
rG4 peaks,0.20 ± 0.59,0.11 ± 0.40,—,—,14.2,8.5,0.5288,0.0,NaN,NaN,0.0,NaN


## 5. Full descriptive statistics table

This summary includes the complete revision-era subgroups for REP (RNA + DNA) and the NBD pipeline. The rG4 and ScanFold sections remain separate because they are not count/coverage features in the same sense as the repeat and motif tables.



In [ ]:
rows = [
    # Transcript length reference row (median±MAD, no statistical test).
    {
        "Group": "NA", "Feature": "Transcript length (median±MAD)",
        "PC counts (mean±std)": _mad_str(pc["transcript_length"]),
        "PC % coverage (mean±std)": "NA",
        "lncRNA counts (mean±std)": _mad_str(lnc["transcript_length"]),
        "lncRNA % coverage (mean±std)": "NA",
        "PC % with hit": np.nan, "lncRNA % with hit": np.nan,
        "VDA (count)": np.nan, "VDA (cov)": np.nan,
        "p_MWU_count": np.nan, "p_MWU_cov": np.nan,
    }
]

# RNA repeats — totals and subgroups.
rows.append(_stats_row("RNA REP", "RNA repeats (total)", pc["rna_global_rm_count"], lnc["rna_global_rm_count"], pc["rna_global_rm_total_length_pct"], lnc["rna_global_rm_total_length_pct"]))
rows.append(_stats_row("RNA TE", "RNA TE (total)", pc["rna_te_count"], lnc["rna_te_count"], pc["rna_te_sum_hit_length_pct"], lnc["rna_te_sum_hit_length_pct"]))
for feat, col in {
    "DNA": "rna_te_dna_count", "LINE": "rna_te_line_count", "LTR": "rna_te_ltr_count",
    "SINE": "rna_te_sine_count", "RC": "rna_te_rc_count", "PLE": "rna_te_ple_count",
    "ERV1": "rna_te_erv1_count", "ERVK": "rna_te_ervk_count", "ERVL": "rna_te_ervl_count",
    "ERVL-MaLR": "rna_te_ervl-malr_count",
}.items():
    rows.append(_stats_row("RNA TE", feat, pc[col], lnc[col]))

rows.append(_stats_row("RNA LCTR", "RNA LCTR (total)", pc["rna_lctr_count"], lnc["rna_lctr_count"], pc["rna_lctr_total_length_pct"], lnc["rna_lctr_total_length_pct"]))
for feat, col in {
    "low complexity": "rna_lctr_low_complexity_count",
    "simple repeat": "rna_lctr_simple_repeat_count",
    "satellite": "rna_lctr_satellite_count",
}.items():
    rows.append(_stats_row("RNA LCTR", feat, pc[col], lnc[col]))

rows.append(_stats_row("RNA pseudogene", "RNA pseudogene (total)", pc["rna_pseudo_count"], lnc["rna_pseudo_count"]))
for feat, col in {
    "rRNA": "rna_pseudo_rrna_count", "scRNA": "rna_pseudo_scrna_count",
    "snRNA": "rna_pseudo_snrna_count", "tRNA": "rna_pseudo_trna_count",
}.items():
    rows.append(_stats_row("RNA pseudogene", feat, pc[col], lnc[col]))

rows.append(_stats_row("NA", "RNA unknown repeats", pc["rna_unknown_count"], lnc["rna_unknown_count"], pc["rna_unknown_total_length_pct"], lnc["rna_unknown_total_length_pct"]))

# DNA repeats — totals and subgroups.
rows.append(_stats_row("DNA REP", "DNA repeats (total)", pc["dna_global_rm_count"], lnc["dna_global_rm_count"], pc["dna_global_rm_total_length_pct"], lnc["dna_global_rm_total_length_pct"]))
rows.append(_stats_row("DNA TE", "DNA TE (total)", pc["dna_te_count"], lnc["dna_te_count"], pc["dna_te_sum_hit_length_pct"], lnc["dna_te_sum_hit_length_pct"]))
for feat, col in {
    "DNA": "dna_te_dna_count", "LINE": "dna_te_line_count", "LTR": "dna_te_ltr_count",
    "SINE": "dna_te_sine_count", "RC": "dna_te_rc_count", "PLE": "dna_te_ple_count",
    "ERV1": "dna_te_erv1_count", "ERVK": "dna_te_ervk_count", "ERVL": "dna_te_ervl_count",
    "ERVL-MaLR": "dna_te_ervl-malr_count",
}.items():
    rows.append(_stats_row("DNA TE", feat, pc[col], lnc[col]))

rows.append(_stats_row("DNA LCTR", "DNA LCTR (total)", pc["dna_lctr_count"], lnc["dna_lctr_count"], pc["dna_lctr_total_length_pct"], lnc["dna_lctr_total_length_pct"]))
for feat, col in {
    "low complexity": "dna_lctr_low_complexity_count",
    "simple repeat": "dna_lctr_simple_repeat_count",
    "satellite": "dna_lctr_satellite_count",
}.items():
    rows.append(_stats_row("DNA LCTR", feat, pc[col], lnc[col]))

rows.append(_stats_row("DNA pseudogene", "DNA pseudogene (total)", pc["dna_pseudo_count"], lnc["dna_pseudo_count"]))
for feat, col in {
    "rRNA": "dna_pseudo_rrna_count", "scRNA": "dna_pseudo_scrna_count",
    "snRNA": "dna_pseudo_snrna_count", "tRNA": "dna_pseudo_trna_count",
}.items():
    rows.append(_stats_row("DNA pseudogene", feat, pc[col], lnc[col]))

rows.append(_stats_row("NA", "DNA unknown repeats", pc["dna_unknown_count"], lnc["dna_unknown_count"], pc["dna_unknown_total_length_pct"], lnc["dna_unknown_total_length_pct"]))

# NBD — totals and motif subgroups.
rows.append(_stats_row("NA", "NBD (total)", pc["total_nonb_count"], lnc["total_nonb_count"], pc["total_nonb_coverage_pct"], lnc["total_nonb_coverage_pct"]))
for feat, cnt_col, cov_col in [
    ("APR",   "apr_hit_count",      "apr_total_length_pct"),
    ("DR",    "dr_hit_count",       "dr_total_length_pct"),
    ("GQ+",   "gq_plus_hit_count",  "gq_plus_total_length_pct"),
    ("GQ-",   "gq_minus_hit_count", "gq_minus_total_length_pct"),
    ("IR",    "ir_hit_count",       "ir_total_length_pct"),
    ("MR",    "mr_hit_count",       "mr_total_length_pct"),
    ("STR",   "str_hit_count",      "str_total_length_pct"),
    ("TRI",   "tri_hit_count",      "tri_total_length_pct"),
    ("Z-DNA", "z_hit_count",        "z_total_length_pct"),
]:
    rows.append(_stats_row("NBD", feat, pc[cnt_col], lnc[cnt_col], pc[cov_col], lnc[cov_col]))

# rG4 peaks (count only; % with hit from binary has_peak column).
rg4 = _stats_row("NA", "rG4 peaks", pc["rg4_peak_count"], lnc["rg4_peak_count"])
rg4["PC % with hit"]     = round(pc["has_peak"].astype(bool).mean() * 100, 1)
rg4["lncRNA % with hit"] = round(lnc["has_peak"].astype(bool).mean() * 100, 1)
rows.append(rg4)

desc = pd.DataFrame(rows)
desc = _apply_bh_fdr(desc, ["p_MWU_count", "p_MWU_cov"])

full_desc_out = desc.copy()
display(full_desc_out.set_index(["Group", "Feature"]).round(4))


PC counts (mean±std)  \
Group          Feature                                               
NA             Transcript length (median±MAD)          1620 ± 1012   
RNA REP        RNA repeats (total)                     1.50 ± 2.55   
RNA TE         RNA TE (total)                          0.76 ± 1.68   
               DNA                                     0.11 ± 0.40   
               LINE                                    0.16 ± 0.52   
               LTR                                     0.09 ± 0.38   
               SINE                                    0.40 ± 1.01   
               RC                                      0.00 ± 0.02   
               PLE                                     0.00 ± 0.01   
               ERV1                                    0.00 ± 0.00   
               ERVK                                    0.00 ± 0.00   
               ERVL                                    0.00 ± 0.00   
               ERVL-MaLR                               0.00 ± 0.00   
RNA LCTR       RNA LCTR (total)                        0.65 ± 1.22   
               low complexity                          0.11 ± 0.37   
               simple repeat                           0.54 ± 1.07   
               satellite                               0.00 ± 0.06   
RNA pseudogene RNA pseudogene (total)                  0.00 ± 0.05   
               rRNA                                    0.00 ± 0.02   
               scRNA                                   0.00 ± 0.02   
               snRNA                                   0.00 ± 0.04   
               tRNA                                    0.00 ± 0.02   
NA             RNA unknown repeats                     0.01 ± 0.70   
DNA REP        DNA repeats (total)                 101.97 ± 187.69   
DNA TE         DNA TE (total)                        24.96 ± 69.62   
               DNA                                    3.26 ± 10.31   
               LINE                                   6.37 ± 20.02   
               LTR                                     2.21 ± 8.56   
               SINE                                  12.97 ± 33.41   
               RC                                      0.01 ± 0.12   
               PLE                                     0.00 ± 0.07   
               ERV1                                    0.00 ± 0.00   
               ERVK                                    0.00 ± 0.00   
               ERVL                                    0.00 ± 0.00   
               ERVL-MaLR                               0.00 ± 0.00   
DNA LCTR       DNA LCTR (total)                      11.17 ± 25.49   
               low complexity                          1.12 ± 2.80   
               simple repeat                         10.00 ± 22.89   
               satellite                               0.06 ± 0.34   
DNA pseudogene DNA pseudogene (total)                  0.20 ± 0.55   
               rRNA                                    0.03 ± 0.20   
               scRNA                                   0.04 ± 0.20   
               snRNA                                   0.11 ± 0.37   
               tRNA                                    0.02 ± 0.17   
NA             DNA unknown repeats                     0.38 ± 1.54   
               NBD (total)                         288.02 ± 551.34   
NBD            APR                                    7.33 ± 17.23   
               DR                                    27.90 ± 56.33   
               GQ+                                    7.76 ± 13.75   
               GQ-                                    7.75 ± 13.96   
               IR                                  124.67 ± 253.92   
               MR                                    35.71 ± 71.63   
               STR                                  60.69 ± 111.92   
               TRI                                    8.18 ± 15.70   
               Z-DNA                                  8.04 ± 18.00   
NA             rG4 peaks                               0

## 6. ScanFold structural metrics

ScanFold outputs thermodynamic/structure metrics (mean ± SD, MWU + VDA).
Same column format as the main table; coverage and presence columns are NA (not applicable).


In [ ]:

# ScanFold thermodynamic/structure metrics — same column format as main table.
# Coverage and presence are not applicable (NA); values go in the counts columns.
SCANFOLD = {
    "avgMFE_mean":      "Avg. MFE (mean)",
    "avgZ_mean":        "Avg. z-score (mean)",
    "avgED_mean":       "Avg. ensemble diversity (mean)",
    "frac_paired":      "Fraction paired",
    "frac_competition": "Fraction competition",
}
sf_rows = []
for col, label in SCANFOLD.items():
    pc_v  = pd.to_numeric(pc[col],  errors="coerce").dropna()
    lnc_v = pd.to_numeric(lnc[col], errors="coerce").dropna()
    n1, n2 = len(pc_v), len(lnc_v)
    u, p = mannwhitneyu(pc_v, lnc_v, alternative="two-sided")
    sf_rows.append({
        "Group": "ScanFold",
        "Feature": label,
        "PC counts (mean±std)": _ms(pc_v, 3),
        "PC % coverage (mean±std)": "NA",
        "lncRNA counts (mean±std)": _ms(lnc_v, 3),
        "lncRNA % coverage (mean±std)": "NA",
        "PC % with hit": np.nan,
        "lncRNA % with hit": np.nan,
        "VDA (count)": _vda(u, n1, n2),
        "VDA (cov)": np.nan,
        "p_MWU_count": p,
        "p_MWU_cov": np.nan,
    })
scanfold = pd.DataFrame(sf_rows)
scanfold = _apply_bh_fdr(scanfold, ["p_MWU_count", "p_MWU_cov"])
display(scanfold.set_index(["Group", "Feature"]).round(4))


PC counts (mean±std)  \
Group    Feature                                               
ScanFold Avg. MFE (mean)                     -31.986 ± 7.243   
         Avg. z-score (mean)                  -0.932 ± 0.333   
         Avg. ensemble diversity (mean)       24.115 ± 1.895   
         Fraction paired                       0.404 ± 0.057   
         Fraction competition                  0.212 ± 0.065   

                                        PC % coverage (mean±std)  \
Group    Feature                                                   
ScanFold Avg. MFE (mean)                                      NA   
         Avg. z-score (mean)                                  NA   
         Avg. ensemble diversity (mean)                       NA   
         Fraction paired                                      NA   
         Fraction competition                                 NA   

                                        lncRNA counts (mean±std)  \
Group    Feature                                                   
ScanFold Avg. MFE (mean)                         -28.604 ± 5.980   
         Avg. z-score (mean)                      -1.016 ± 0.384   
         Avg. ensemble diversity (mean)           23.783 ± 2.145   
         Fraction paired                           0.378 ± 0.058   
         Fraction competition                      0.188 ± 0.066   

                                        lncRNA % coverage (mean±std)  \
Group    Feature                                                       
ScanFold Avg. MFE (mean)                                          NA   
         Avg. z-score (mean)                                      NA   
         Avg. ensemble diversity (mean)                           NA   
         Fraction paired                                          NA   
         Fraction competition                                     NA   

                                         PC % with hit  lncRNA % with hit  \
Group    Feature                                                            
ScanFold Avg. MFE (mean)                           NaN                NaN   
         Avg. z-score (mean)                       NaN                NaN   
         Avg. ensemble diversity (mean)            NaN                NaN   
         Fraction paired                           NaN                NaN   
         Fraction competition                      NaN                NaN   

                                         VDA (count)  VDA (cov)  p_MWU_count  \
Group    Feature                                                               
ScanFold Avg. MFE (mean)                      0.3667        NaN          0.0   
         Avg. z-score (mean)                  0.5675        NaN          0.0   
         Avg. ensemble diversity (mean)       0.5508        NaN          0.0   
         Fraction paired                      0.6343        NaN          0.0   
         Fraction competition                 0.6051        NaN          0.0   

                                         p_MWU_cov  p_MWU_count_adj  \
Group    Feature                                                      
ScanFold Avg. MFE (mean)                       NaN              0.0   
         Avg. z-score (mean)                   NaN              0.0   
         Avg. ensemble diversity (mean)        NaN              0.0   
         Fraction paired                       NaN              0.0   
         Fraction competition                  NaN              0.0   

                                         p_MWU_cov_adj  
Group    Feature                                        
ScanFold Avg. MFE (mean)                           NaN  
         Avg. z-score (mean)                       NaN  
         Avg. ensemble diversity (mean)            NaN  
         Fraction paired                           NaN  
         Fraction competition                      NaN

## 7. Export


In [ ]:

desc_out.to_csv(TABLE_OUT / "Table2_element_stats.csv", index=False)
full_desc_out.to_csv(TABLE_OUT / "S12a_extended_element_stats.csv", index=False)
scanfold.to_csv(TABLE_OUT / "S12b_scanfold_structure_stats.csv", index=False)
print("Wrote:")
print(" ", TABLE_OUT / "S12a_extended_element_stats.csv", full_desc_out.shape)
print(" ", TABLE_OUT / "S12b_scanfold_structure_stats.csv", scanfold.shape)


Wrote:
  /mnt/cbib/LNClassifier/paper/revision-wt/results/gencode.v47.common.cdhit.cv/supplementary_tables/element_statistics_tables/S12a_extended_element_stats.csv (56, 14)
  /mnt/cbib/LNClassifier/paper/revision-wt/results/gencode.v47.common.cdhit.cv/supplementary_tables/element_statistics_tables/S12b_scanfold_structure_stats.csv (5, 14)
